# 📊 Automated Weekly Business Report

End-to-end pipeline that:
- Pulls data from PostgreSQL databases (or runs on **mock data** for demo)
- Computes **WoW** (Week-over-Week) and **YoY** (Year-over-Year) metrics
- Generates styled HTML tables with color-coded highlights
- Emails the report to stakeholders automatically

---
**▶ To run this notebook without any setup:**
```
USE_MOCK_DATA = True   ← already set, just run all cells
SEND_EMAIL    = False  ← saves HTML to file instead of sending
```

**▶ To connect real data / email:**
1. Copy `.env.example` → `.env` and fill in your credentials
2. Set `USE_MOCK_DATA = False` and/or `SEND_EMAIL = True`

## ⚙️ Configuration

In [ ]:
# ── SWITCHES ────────────────────────────────────────────────────
USE_MOCK_DATA = True   # True = demo mode (no DB needed)
SEND_EMAIL    = False  # True = sends real email (requires .env)
# ────────────────────────────────────────────────────────────────

import os
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # dotenv optional; values fall back to defaults below

DB_CONFIG = {
    "analytics_db": {
        "host":     os.getenv("DB_HOST",             "localhost"),
        "port":     os.getenv("DB_PORT",             "5432"),
        "db":       os.getenv("DB_NAME_ANALYTICS",   "analytics"),
        "user":     os.getenv("DB_USER",             "db_user"),
        "password": os.getenv("DB_PASSWORD",         ""),
    },
    "main_db": {
        "host":     os.getenv("DB_HOST",             "localhost"),
        "port":     os.getenv("DB_PORT",             "5432"),
        "db":       os.getenv("DB_NAME_MAIN",        "main"),
        "user":     os.getenv("DB_USER",             "db_user"),
        "password": os.getenv("DB_PASSWORD",         ""),
    },
}

EMAIL_CONFIG = {
    "sender":      os.getenv("SENDER_EMAIL",       "your_email@gmail.com"),
    "password":    os.getenv("GMAIL_APP_PASSWORD", ""),
    "recipients":  os.getenv("RECIPIENT_EMAILS",   "recipient@example.com").split(","),
    "smtp_server": "smtp.gmail.com",
    "smtp_port":   465,
}

print("Config loaded")
print(f"  Mode  : {'Mock Data (demo)' if USE_MOCK_DATA else 'Live Database'}")
print(f"  Email : {'Will send' if SEND_EMAIL else 'Save to file only'}")

## 📦 Install Dependencies

In [ ]:
import subprocess, sys

packages = [
    "sqlalchemy", "psycopg2-binary", "pandas",
    "premailer", "matplotlib", "jinja2", "python-dotenv"
]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All packages ready")

## 📅 Date Range Setup

In [ ]:
import warnings
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from calendar import month_name, monthrange
warnings.simplefilter(action='ignore')

yesterday    = datetime.now() - timedelta(days=1)
end_date     = yesterday.strftime("%Y%m%d")
start_date   = (yesterday - timedelta(days=27)).strftime("%Y%m%d")
start_date2  = (yesterday + timedelta(days=2)).strftime("%Y%m%d")
start_date1  = yesterday.strftime("%Y%m") + "01"
end_date1    = end_date
day_of_month = int(end_date[-2:])

print(f"WoW window : {start_date}  ->  {end_date}")
print(f"YoY window : {start_date1} ->  {end_date1}")

## 🗄️ Data Layer — Live DB or Mock Data

In [ ]:
def get_live_data():
    """
    Connect to real PostgreSQL databases and fetch reporting data.
    Requires DB_HOST, DB_USER, DB_PASSWORD etc. set in .env
    """
    from sqlalchemy import create_engine

    engines = {}
    for name, cfg in DB_CONFIG.items():
        url = (f"postgresql+psycopg2://{cfg['user']}:{cfg['password']}"
               f"@{cfg['host']}:{cfg['port']}/{cfg['db']}")
        engines[name] = create_engine(url)
        print(f"Connected to {name}")

    # WoW daily stats
    sql_wow = f"""
        SELECT stats_date, flow_id,
               COALESCE(SUM(cost), 0)                        AS cost,
               COALESCE(SUM(sell_revenue), 0)                AS revenue,
               COALESCE(SUM(sell_revenue) - SUM(cost), 0)    AS profit
        FROM   reporting.daily_stats
        WHERE  stats_date BETWEEN '{start_date}' AND '{end_date}'
        GROUP  BY stats_date, flow_id
    """
    # Flow metadata (routes, sources)
    sql_routes = """
        SELECT flow_id,
               traffic_route,
               buy_side_name  AS buy_source,
               sell_side_name AS sell_source
        FROM   api.flows_view
        WHERE  team = 'BT3'
    """
    # Month-to-date profit
    sql_profit = f"""
        SELECT stats_date, flow_id,
               COALESCE(SUM(cost), 0)                     AS cost,
               COALESCE(SUM(sell_revenue), 0)             AS revenue,
               SUM(sell_revenue) - SUM(cost)              AS profit
        FROM   reporting.daily_stats
        WHERE  stats_date BETWEEN '{start_date1}' AND '{end_date1}'
        GROUP  BY stats_date, flow_id
    """
    # Historical YoY table (pre-aggregated by month)
    sql_yoy = "SELECT * FROM reporting.yoy_performance"

    with engines["analytics_db"].connect() as conn:
        df_stats  = pd.read_sql_query(sql_wow,    conn)
        df_profit = pd.read_sql_query(sql_profit, conn)
        df_yoy    = pd.read_sql_query(sql_yoy,    conn)
    with engines["main_db"].connect() as conn:
        df_routes = pd.read_sql_query(sql_routes, conn)

    return df_stats, df_routes, df_profit, df_yoy


def get_mock_data():
    """
    Generate realistic dummy data that mirrors the live DB schema.
    Used for demos and portfolio — no database connection needed.
    """
    np.random.seed(42)

    BUY_SOURCES    = ["Google", "Bing", "Yahoo", "DuckDuckGo", "Baidu"]
    SELL_SOURCES   = ["Publisher A", "Publisher B", "Publisher C", "Ginsu", "Publisher D"]
    TRAFFIC_ROUTES = ["Route-US", "Route-EU", "Route-APAC", "Route-LATAM"]
    FLOW_IDS       = list(range(1, 21))

    # --- Flow metadata ---
    df_routes = pd.DataFrame({
        "flow_id":       FLOW_IDS,
        "traffic_route": np.random.choice(TRAFFIC_ROUTES, size=20),
        "buy_source":    np.random.choice(BUY_SOURCES,    size=20),
        "sell_source":   np.random.choice(SELL_SOURCES,   size=20),
    })

    # --- 28-day WoW stats ---
    rows = []
    for d in range(28):
        date = (yesterday - timedelta(days=d)).strftime("%Y%m%d")
        for fid in FLOW_IDS:
            cost    = np.random.uniform(5_000, 50_000)
            revenue = cost * np.random.uniform(1.05, 1.40)
            rows.append({"stats_date": date, "flow_id": fid,
                         "cost": cost, "revenue": revenue,
                         "profit": revenue - cost})
    df_stats = pd.DataFrame(rows)

    # --- Month-to-date profit ---
    rows2 = []
    for d in range(day_of_month):
        date = (yesterday - timedelta(days=d)).strftime("%Y%m%d")
        for fid in FLOW_IDS:
            cost    = np.random.uniform(5_000, 50_000)
            revenue = cost * np.random.uniform(1.05, 1.40)
            rows2.append({"stats_date": date, "flow_id": fid,
                          "cost": cost, "revenue": revenue,
                          "profit": revenue - cost})
    df_profit = pd.DataFrame(rows2)

    # --- Historical YoY (last 8 years + current year completed months) ---
    current_year = datetime.now().year
    current_month_num = datetime.now().month
    yoy_rows = []
    for yr in range(current_year - 7, current_year + 1):
        for mo_num, mo_name in enumerate(month_name):
            if not mo_name:
                continue
            # For current year, only include fully completed months
            if yr == current_year and mo_num >= current_month_num:
                continue
            base_cost    = np.random.uniform(3e6, 8e6)
            base_revenue = base_cost * np.random.uniform(1.08, 1.35)
            profit       = base_revenue - base_cost
            yoy_rows.append({
                "year":           yr,
                "month":          mo_name,
                "cost":           base_cost,
                "revenue":        base_revenue,
                "profit":         profit,
                "profit_percent": profit / base_cost,
            })
    df_yoy = pd.DataFrame(yoy_rows)

    print("Mock data generated")
    print(f"  df_stats  : {df_stats.shape[0]} rows")
    print(f"  df_routes : {df_routes.shape[0]} flows")
    print(f"  df_profit : {df_profit.shape[0]} rows")
    print(f"  df_yoy    : {df_yoy.shape[0]} rows")
    return df_stats, df_routes, df_profit, df_yoy


# ── Load data ────────────────────────────────────────────────────
if USE_MOCK_DATA:
    df_stats, df_routes, df_profit_raw, YoY_report = get_mock_data()
else:
    df_stats, df_routes, df_profit_raw, YoY_report = get_live_data()

## 🔧 Data Transformation — WoW

In [ ]:
# Standardise YoY column names
YoY_report.rename(columns={
    'year': 'Year', 'month': 'Month',
    'profit': 'Profit', 'cost': 'Cost',
    'revenue': 'Revenue', 'profit_percent': 'Profit%'
}, inplace=True)

# Join daily stats with flow metadata
df          = df_stats.merge(df_routes, on='flow_id', how='inner')
df_on_profit = df_profit_raw.merge(df_routes, on='flow_id', how='inner')

for frame in [df, df_on_profit]:
    frame.fillna({'profit': 0, 'cost': 0, 'revenue': 0}, inplace=True)
    frame['stats_date'] = pd.to_datetime(frame['stats_date'], format='%Y%m%d')
    # Anchor each date to the Monday of its week
    frame['Week'] = frame['stats_date'] - pd.to_timedelta(
        frame['stats_date'].dt.weekday, unit='d')

# Weekly aggregation
GROUP = ['Week', 'traffic_route', 'buy_source', 'sell_source']
AGG   = {'cost': 'sum', 'revenue': 'sum', 'profit': 'sum'}
df           = df.groupby(GROUP).agg(AGG).reset_index()
df_on_profit = df_on_profit.groupby(GROUP).agg(AGG).reset_index()

# Remove excluded / internal sources
EXCLUDED_BUY  = ['Internal', 'Unassigned Revenue', 'Test Source']
EXCLUDED_SELL = ['Default Source']
for frame in [df, df_on_profit]:
    frame.drop(frame[frame['buy_source'].isin(EXCLUDED_BUY)].index,   inplace=True)
    frame.drop(frame[frame['sell_source'].isin(EXCLUDED_SELL)].index, inplace=True)

print(f"WoW data ready — {df.shape[0]} rows, {df['Week'].nunique()} weeks")
df.head(3)

## 📐 WoW Pivot Tables

In [ ]:
# Rename for display
df.rename(columns={
    'traffic_route': 'Traffic Route', 'buy_source': 'Buy Source',
    'sell_source': 'Sell Source',
    'cost': 'Cost', 'revenue': 'Revenue', 'profit': 'Profit'
}, inplace=True)

# Partner-specific profit adjustment (50% share)
df['Adjusted Profit'] = df.apply(
    lambda r: r['Profit'] * 0.5 if r['Sell Source'] == 'Ginsu' else r['Profit'], axis=1)

df['Profit Rounded']  = df['Adjusted Profit'].apply(lambda x: round(x / 1000) * 1000)
df['Cost Rounded']    = df['Cost'].apply(lambda x: round(x / 1000) * 1000)
df['Revenue Rounded'] = df['Revenue'].apply(lambda x: round(x / 1000) * 1000)
df['Profit%']         = (
    (df['Profit Rounded'] / df['Cost Rounded'] * 100)
    .replace([float('inf'), -float('inf')], 0).fillna(0).round(0)
)

weeks_sorted = sorted(df['Week'].dropna().unique(), reverse=True)


def parse_value(val):
    """Parse formatted strings like '$12K' or '$1.5M' back to float."""
    if val == '' or pd.isna(val):
        return 0
    s = str(val).replace('$', '')
    if 'M' in s:
        return float(s.replace('M', '')) * 1_000_000
    if 'K' in s:
        return float(s.replace('K', '')) * 1_000
    return float(s) if s else 0


def make_pivot(metric_name, col, fmt_fn, group_col, override=None):
    raw = (override if override is not None
           else df.pivot_table(index=group_col, columns='Week',
                               values=col, aggfunc='sum'))
    raw = raw.reindex(columns=weeks_sorted, fill_value=0).applymap(fmt_fn)
    raw.columns = pd.MultiIndex.from_tuples(
        [(metric_name, w.strftime('%d-%b')) for w in weeks_sorted])
    return raw


def generate_wow_table(group_col):
    fmt_k   = lambda x: f"${int(x/1000)}K" if pd.notnull(x) else ""
    fmt_pct = lambda x: f"{int(x)}%"        if pd.notnull(x) else ""

    profit_raw  = df.pivot_table(index=group_col, columns='Week',
                                  values='Profit Rounded',  aggfunc='sum')
    cost_raw    = df.pivot_table(index=group_col, columns='Week',
                                  values='Cost Rounded',    aggfunc='sum')
    revenue_raw = df.pivot_table(index=group_col, columns='Week',
                                  values='Revenue Rounded', aggfunc='sum')
    pct_raw     = (profit_raw / cost_raw * 100)\
                  .replace([float('inf'), -float('inf')], 0).fillna(0).round(0)

    table = pd.concat([
        make_pivot('Profit',  'Profit Rounded',  fmt_k,   group_col),
        make_pivot('Cost',    'Cost Rounded',    fmt_k,   group_col),
        make_pivot('Revenue', 'Revenue Rounded', fmt_k,   group_col),
        make_pivot('Profit%', None,              fmt_pct, group_col, override=pct_raw),
    ], axis=1)

    # Sort by latest week profit descending
    table['__sort'] = profit_raw[weeks_sorted[0]].reindex(table.index)
    table = table.sort_values('__sort', ascending=False).drop(columns='__sort').reset_index()

    # Grand Total row
    fmt_m   = lambda v: f"${round(v/1e6,2)}M" if pd.notnull(v) else ""
    pct_str = lambda p, c: f"{int(round(p/c*100))}%" if c else ""

    total = ["Grand Total"]
    total += [fmt_m(profit_raw[w].sum())  for w in weeks_sorted]
    total += [fmt_m(cost_raw[w].sum())    for w in weeks_sorted]
    total += [fmt_m(revenue_raw[w].sum()) for w in weeks_sorted]
    total += [pct_str(profit_raw[w].sum(), cost_raw[w].sum()) for w in weeks_sorted]
    table.loc[len(table)] = total
    return table


traffic_report = generate_wow_table('Traffic Route')
buy_report     = generate_wow_table('Buy Source')
sell_report    = generate_wow_table('Sell Source')

# Drop low-volume traffic routes (< $20K cost in latest week)
latest_cost_col = ('Cost', weeks_sorted[0].strftime('%d-%b'))
if latest_cost_col in traffic_report.columns:
    keep = traffic_report[latest_cost_col].apply(parse_value) >= 20_000
    keep |= traffic_report['Traffic Route'] == 'Grand Total'
    traffic_report = traffic_report[keep].reset_index(drop=True)

print("WoW pivot tables ready")
traffic_report.head()

## 📈 YoY Calculation

In [ ]:
month_to_num = {name: num for num, name in enumerate(month_name) if name}
num_to_month = {num: name for num, name in enumerate(month_name) if name}

YoY_report['Month_num'] = YoY_report['Month'].apply(lambda m: month_to_num.get(m, 0))
YoY_report['Year']      = YoY_report['Year'].astype(int)

# Rename for merge alignment
df_on_profit.rename(columns={
    'traffic_route': 'Traffic Route', 'buy_source': 'Buy Source',
    'sell_source': 'Sell Source'
}, inplace=True, errors='ignore')

# Apply 50% profit share for partner sell source
df_on_profit['profit'] = df_on_profit.apply(
    lambda r: r['profit'] * 0.5 if r['Sell Source'] == 'Ginsu' else r['profit'], axis=1)

total_cost    = df_on_profit['cost'].sum()
total_revenue = df_on_profit['revenue'].sum()
total_profit  = df_on_profit['profit'].sum()

print(f"Month-to-date  Cost:    ${total_cost:>15,.0f}")
print(f"Month-to-date  Revenue: ${total_revenue:>15,.0f}")
print(f"Month-to-date  Profit:  ${total_profit:>15,.0f}")

year  = int(end_date[:4])
month = int(end_date[4:6])

condition = (YoY_report['Year'] == year) & (YoY_report['Month_num'] == month)
if YoY_report.loc[condition].empty:
    new_row = pd.DataFrame({'Year': [year], 'Month_num': [month],
                             'Month': [num_to_month[month]],
                             'Cost': [0], 'Revenue': [0],
                             'Profit': [0], 'Profit%': [0]})
    YoY_report = pd.concat([YoY_report, new_row], ignore_index=True)
    condition  = (YoY_report['Year'] == year) & (YoY_report['Month_num'] == month)

# Extrapolate to full-month if mid-month
total_days_in_month = monthrange(year, month)[1]
estimated_profit    = (
    (total_profit / day_of_month) * total_days_in_month
    if day_of_month != total_days_in_month else total_profit
)

YoY_report.loc[condition, ['Cost', 'Revenue', 'Profit', 'Profit%']] = [
    total_cost, total_revenue, estimated_profit,
    total_profit / total_cost if total_cost else 0
]
YoY_report.sort_values(['Year', 'Month_num'], inplace=True)

print(f"\nEstimated full-month profit ({num_to_month[month]} {year}): "
      f"${estimated_profit/1e6:.2f}M")

## 🗂️ YoY Pivot Table

In [ ]:
pivot = pd.pivot_table(
    YoY_report,
    values=['Profit', 'Cost', 'Revenue', 'Profit%'],
    index='Month', columns='Year'
)
pivot.columns.set_names(['Metric', 'Year'], inplace=True)
pivot = pivot.reorder_levels(['Year', 'Metric'], axis=1)
pivot = pivot.sort_index(axis=1, level='Year', ascending=False)

profit_s  = pivot.xs('Profit',  level='Metric', axis=1).sum()
cost_s    = pivot.xs('Cost',    level='Metric', axis=1).sum()
revenue_s = pivot.xs('Revenue', level='Metric', axis=1).sum()
pct_s     = (profit_s / cost_s).fillna(0)

total_row = pd.Series(index=pivot.columns, dtype='float64')
for yr in pivot.columns.get_level_values('Year').unique():
    total_row[(yr, 'Profit')]  = profit_s.get(yr, 0)
    total_row[(yr, 'Cost')]    = cost_s.get(yr, 0)
    total_row[(yr, 'Revenue')] = revenue_s.get(yr, 0)
    total_row[(yr, 'Profit%')] = pct_s.get(yr, 0)

pivot_with_total = pivot.copy()
pivot_with_total.loc['Total'] = total_row
pivot_with_total = pivot_with_total.reset_index()

MONTH_ORDER = ["January","February","March","April","May","June",
               "July","August","September","October","November",
               "December","Total"]

pivot_with_total["Month"] = pd.Categorical(
    pivot_with_total["Month"].astype(str).str.strip().str.title(),
    categories=MONTH_ORDER, ordered=True)
pivot_with_total = pivot_with_total.sort_values("Month")

# Fix current-year total Profit% with estimated profit
current_year = datetime.now().year
if current_year in pivot_with_total.columns.get_level_values('Year'):
    adj_p   = profit_s.get(current_year, 0) - estimated_profit + total_profit
    adj_pct = adj_p / cost_s.get(current_year, 1)
    pivot_with_total.loc[
        pivot_with_total['Month'] == 'Total', (current_year, 'Profit%')
    ] = adj_pct

print("YoY pivot ready")
pivot_with_total.head(3)

## 🎨 Format & Style Tables

In [ ]:
end_dt        = datetime.strptime(end_date, "%Y%m%d")
current_month = end_dt.strftime("%B")

# Detect the actual month column — could be 'Month' or ('Month','') depending on pandas version
month_col = next(
    c for c in pivot_with_total.columns
    if (c == 'Month') or (isinstance(c, tuple) and c[0] == 'Month')
)

# ── Format YoY cells ─────────────────────────────────────────────
def format_yoy_cell(val, metric, row_month, yr):
    if pd.isna(val):
        return ""
    # Mark current month's profit as estimated with asterisk
    if metric == "Profit" and str(row_month) == current_month and yr == current_year:
        return f"${val/1e6:.2f}*M"
    if "Profit%" in str(metric):
        return f"{val*100:.2f}%"
    return f"${val/1e6:.2f}M"

formatted_pivot = pivot_with_total.copy()
for col in formatted_pivot.columns:
    # Skip the month column and any non-(year, metric) columns
    if not (isinstance(col, tuple) and isinstance(col[0], int)):
        continue
    yr, metric = col
    formatted_pivot[col] = formatted_pivot.apply(
        lambda row, c=col, y=yr, m=metric: format_yoy_cell(
            row[c], m, row[month_col], y), axis=1)

# Column order: Profit, Cost, Revenue, Profit% per year
METRIC_ORDER = ["Profit", "Cost", "Revenue", "Profit%"]
years_sorted = sorted(
    {c[0] for c in formatted_pivot.columns
     if isinstance(c, tuple) and isinstance(c[0], int)},
    reverse=True)
final_cols = [month_col] + [
    (yr, m) for yr in years_sorted for m in METRIC_ORDER
    if (yr, m) in formatted_pivot.columns
]
formatted_pivot = formatted_pivot[final_cols]
formatted_pivot[month_col] = pd.Categorical(
    formatted_pivot[month_col].astype(str),
    categories=MONTH_ORDER, ordered=True)
formatted_pivot = formatted_pivot.sort_values(month_col)

# ── Shared table style ────────────────────────────────────────────
BASE_STYLES = [
    {'selector': 'table', 'props': [
        ('border-collapse','collapse'), ('font-family','Arial,sans-serif'),
        ('font-size','12px'), ('width','auto'), ('max-width','90%')]},
    {'selector': 'th', 'props': [
        ('background-color','#f4f4f4'), ('color','black'), ('font-weight','bold'),
        ('border','1px solid black'), ('text-align','center'), ('padding','3px')]},
    {'selector': 'td', 'props': [
        ('border','1px solid black'), ('text-align','right'),
        ('font-size','12px'), ('padding','3px')]},
    {'selector': 'td.col0', 'props': [('text-align','left')]},
]

def group_borders(df):
    groups, borders = {}, []
    for i, c in enumerate(df.columns):
        groups.setdefault(c[0] if isinstance(c, tuple) else c, []).append(i)
    for cols in groups.values():
        if cols[0] != 0:
            borders.append({'selector': f'th.col{cols[0]}, td.col{cols[0]}',
                             'props': [('border-left','2px solid black')]})
    return borders


# ── WoW styling ───────────────────────────────────────────────────
def first_col_bg(val):
    return 'background-color: #efefef; font-weight: bold'

def grand_total_style(row):
    return (['background-color: #666666; color: white'] * len(row)
            if str(row.iloc[0]) == 'Grand Total' else [''] * len(row))

def highlight_wow_profit(row):
    """Green if latest week profit is highest; red if lowest (>=5% change)."""
    if str(row.iloc[0]) == 'Grand Total':
        return [''] * len(row)
    pcols  = [c for c in row.index if c[0] == 'Profit']
    if len(pcols) < 2:
        return [''] * len(row)
    vals   = [parse_value(row[c]) for c in pcols]
    latest = vals[0]
    prev   = vals[1]
    styles = [''] * len(row)
    if all(v == 0 for v in vals):
        return styles
    diff = abs((latest - prev) / prev * 100) if prev else float('inf')
    if diff >= 5:
        idx = row.index.get_loc(pcols[0])
        if   latest == max(vals): styles[idx] = 'background-color: #d9ead3'  # green
        elif latest == min(vals): styles[idx] = 'background-color: #f4cccc'  # red
    return styles


def style_wow(report_df):
    return (
        report_df.style
        .applymap(first_col_bg, subset=pd.IndexSlice[:, report_df.columns[0]])
        .apply(grand_total_style,   axis=1)
        .apply(highlight_wow_profit, axis=1)
        .set_table_styles(BASE_STYLES + group_borders(report_df))
        .set_table_attributes('class="dataframe" style="border:2px solid black;"')
        .hide(axis='index')
    )


# ── YoY styling ───────────────────────────────────────────────────
def style_yoy(fmt_df, raw_df):
    pcols = [c for c in fmt_df.columns if c[1] == 'Profit']

    def highlight_best_year(row):
        """Highlight the year with highest profit per month in green."""
        if str(row[month_col]) == 'Total':
            return [''] * len(row)
        styles   = [''] * len(row)
        raw_vals = raw_df.loc[row.name, pcols].astype(float)
        best_col = raw_vals.idxmax()
        for i, c in enumerate(fmt_df.columns):
            if c == best_col:
                styles[i] = 'background-color: #55c991'
        return styles

    def total_row_style(row):
        return (['background-color: #666666; color: white; font-weight: bold'] * len(row)
                if str(row[month_col]) == 'Total' else [''] * len(row))

    return (
        fmt_df.style
        .apply(highlight_best_year, axis=1)
        .apply(total_row_style,     axis=1)
        .set_table_styles(BASE_STYLES + group_borders(fmt_df))
        .set_table_attributes(
            'class="dataframe" style="border:3px solid black; border-collapse:collapse;"')
        .hide(axis='index')
    )


html_traffic = style_wow(traffic_report).to_html(index=False)
html_buy     = style_wow(buy_report).to_html(index=False)
html_sell    = style_wow(sell_report).to_html(index=False)
html_yoy     = style_yoy(formatted_pivot, pivot_with_total).to_html(index=False)

print("HTML tables rendered")

## 📧 Build Email Body

In [ ]:
profit_M     = round(total_profit    / 1e6, 2)
est_profit_M = round(estimated_profit / 1e6, 2)
month_str    = num_to_month[month]

html_body = f"""
<html><head><style>
  * {{ margin:0; padding:0; }}
  body {{ padding:8px; font-family:Arial,sans-serif; }}
  p    {{ font-size:13px; margin:2px 0; }}
  h4   {{ margin:8px 0 3px; }}
  table {{ border-collapse:collapse; font-size:12px; }}
  ul   {{ font-size:14px; list-style-type:disc; margin-left:20px; }}
</style></head><body>

  <p>Hi Team,</p><br>

  <ul><li><strong>YoY Stats — last 8 years</strong></li></ul>
  {html_yoy}
  <p><em>
    *Estimated profit for {month_str} {year}: ${est_profit_M}M &nbsp;|
    &nbsp; Actual so far ({day_of_month} days): ${profit_M}M
  </em></p>
  <p><em>*Profit for partner sell source is calculated at 50% of Gross Profit.</em></p>
  <br>

  <h4>Buy Source Stats</h4>
  {html_buy}<br>

  <h4>Sell Source Stats</h4>
  {html_sell}<br>

  <h4>Traffic Route Stats</h4>
  {html_traffic}
  <p><em>*Low-volume routes (cost &lt; $20K/week) excluded.</em></p>
  <br>

  <p>Thanks,<br>Your Name</p>
</body></html>
"""

from premailer import transform
email_html = transform(f"<html><body>{html_body}</body></html>")

# Always save preview
with open('email_preview.html', 'w', encoding='utf-8') as f:
    f.write(html_body)

print("Email HTML ready — open email_preview.html to review")

## 📤 Send Email

In [ ]:
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

def build_subject():
    def ordinal(n):
        return "%d%s" % (n, "th" if 4 <= n%100 <= 20
                         else {1:"st",2:"nd",3:"rd"}.get(n%10, "th"))
    e = datetime.now() - timedelta(days=1)
    s = e - timedelta(days=6)
    return (f"Weekly Newsletter "
            f"({ordinal(s.day)} {s.strftime('%b')} – "
            f"{ordinal(e.day)} {e.strftime('%b')})")

if SEND_EMAIL:
    msg = MIMEMultipart("alternative")
    msg["Subject"] = build_subject()
    msg["From"]    = EMAIL_CONFIG["sender"]
    msg["To"]      = ", ".join(EMAIL_CONFIG["recipients"])
    msg.attach(MIMEText(email_html, "html"))

    try:
        with smtplib.SMTP_SSL(EMAIL_CONFIG["smtp_server"],
                               EMAIL_CONFIG["smtp_port"]) as server:
            server.login(EMAIL_CONFIG["sender"], EMAIL_CONFIG["password"])
            server.sendmail(EMAIL_CONFIG["sender"],
                            EMAIL_CONFIG["recipients"],
                            msg.as_string())
        print(f"Email sent! — {msg['Subject']}")
    except Exception as e:
        print(f"Email failed: {e}")
else:
    print("SEND_EMAIL=False")
    print(f"Report saved to: email_preview.html")
    print(f"Subject would be: {build_subject()}")
    print()
    print("To send for real:")
    print("  1. Copy .env.example -> .env")
    print("  2. Fill in your Gmail + DB credentials")
    print("  3. Set SEND_EMAIL = True in the first cell")